## Importing Libraries

In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import clickhouse_connect

### Define Parameters

## In the code below remember to repeat all months from August 2025 todate so that cloned files are updated, also truncate all tables in clickhouse. ‼️‼️‼️‼️‼️‼️‼️‼️‼️‼️‼️‼️‼️‼️‼️‼️

In [2]:
MONTH = pd.Period("2026-05")  # <-- only thing to change each run

START, END = MONTH.start_time.date(), (MONTH + 1).start_time.date()

def date_filter(col: str) -> str:
    """SQL WHERE clause for the analysis month, on the given timestamp column."""
    return f"WHERE {col} >= '{START}' AND {col} < '{END}'"

# KYC dump is for the same month as the IMEI data
KYC_TAG = MONTH.strftime("%m_%y")          # e.g. "06_26"
MONTH_TAG = MONTH.strftime("%Y-%m")        # e.g. "2026-06"

KYC_DIR = Path("/Volumes/E$/KYC/Merged Clean Dumps/2026")
OUT_DIR = Path("/Volumes/E$/CEIR/Clean Dumps")
MCC_CSV = Path("/Users/wmuheki/Documents/Projects/Analytics/ceir/clean_dumps/MCC_Each_country.csv")

print(date_filter("imei_first_seen"))
print(f"KYC files: df_{KYC_TAG}.parquet / df_NID_{KYC_TAG}.parquet")

WHERE imei_first_seen >= '2026-05-01' AND imei_first_seen < '2026-06-01'
KYC files: df_05_26.parquet / df_NID_05_26.parquet


### Define Clickhouse Connect - Single Reused

In [3]:
client = clickhouse_connect.get_client(
    host='192.168.1.95',
    port=8123,
    username='default',
    password='',
    settings={
        'max_memory_usage': 4000000000,  # 4GB max
        'max_threads': 2,
        'priority': 5
    }
)

### Helper Functions

In [4]:
def attach_kyc(target_df: pd.DataFrame, nid_df: pd.DataFrame, fallback_df: pd.DataFrame) -> pd.DataFrame:
    """Attach KYC on msisdn: NID-registered values win, fallback fills the gaps."""
    out = target_df.merge(nid_df, on='msisdn', how='left')
    out = out.merge(fallback_df, on='msisdn', how='left', suffixes=('', '_fb'))

    overlap = ['id_type', 'id_number', 'prefix', 'mno']  # columns present in both KYC frames
    for col in overlap:
        out[col] = out[col].astype('object').fillna(out[f'{col}_fb'].astype('object'))

    out = out.drop(columns=[f'{col}_fb' for col in overlap])

    for col in ['id_type', 'prefix', 'mno']:
        out[col] = out[col].astype('category')
    return out


IMSI_PREFIX_MNO = {
    '64110': 'MTN',
    '64120': 'HAMILTON',
    '64108': 'TALKIO',
    '64101': 'AIRTEL',
    '64122': 'AIRTEL',
}

def fill_mno_from_imsi(df: pd.DataFrame) -> pd.DataFrame:
    """Where mno is missing, derive it from the IMSI prefix."""
    mask = df['mno'].isna()
    imsi = df['imsi'].astype('string')

    if isinstance(df['mno'].dtype, pd.CategoricalDtype):
        new_cats = [m for m in set(IMSI_PREFIX_MNO.values()) if m not in df['mno'].cat.categories]
        if new_cats:
            df['mno'] = df['mno'].cat.add_categories(new_cats)

    for prefix, operator in IMSI_PREFIX_MNO.items():
        df.loc[mask & imsi.str.startswith(prefix), 'mno'] = operator
    return df


def add_country(df: pd.DataFrame, mcc_lookup: pd.DataFrame) -> pd.DataFrame:
    """Extract MCC (first 3 digits of IMSI) and merge in the country name."""
    df['imsi'] = df['imsi'].astype('string')
    df['mcc'] = (
        df['imsi']
        .str.replace(r'\D+', '', regex=True)  # keep digits only
        .str.slice(0, 3)
    )
    df = df.merge(mcc_lookup[['mcc', 'country']], how='left', on='mcc')
    df['country'] = df['country'].fillna('UNKNOWN')
    return df


def save_monthly_parquet(df: pd.DataFrame, category: str) -> Path:
    """Save df to <OUT_DIR>/<category>/<category.lower()>_<YYYY-MM>.parquet."""
    out_path = OUT_DIR / category / f"{category.lower()}_{MONTH_TAG}.parquet"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(out_path)
    print(f"Saved {len(df):,} rows -> {out_path}")
    return out_path

### Importing KYC data

In [5]:
df = pd.read_parquet(KYC_DIR / f"df_{KYC_TAG}.parquet")
df_NID = pd.read_parquet(KYC_DIR / f"df_NID_{KYC_TAG}.parquet")

In [6]:
df.head()

,msisdn,first_name,surname,id_type,id_number,prefix,mno
0,0071607211,NUBUWATI,LWANGA,NATIONAL_ID,CF92098105FTWE,071,UTCL
1,0071607212,NUBUWATI,LWANGA,NATIONAL_ID,CF92098105FTWE,071,UTCL
2,0071607213,NUBUWATI,LWANGA,NATIONAL_ID,CF92098105FTWE,071,UTCL
3,0071607214,NUBUWATI,LWANGA,NATIONAL_ID,CF92098105FTWE,071,UTCL
4,0411671910,DAVID,WASSWA,NATIONAL_ID,CM7305210F16FL,04,UTCL


In [7]:
df_NID.head()

,msisdn,first_name,surname,id_type,id_number,prefix,mno,gender,birth_year,age,district
0,0071607211,NUBUWATI,LWANGA,NATIONAL_ID,CF92098105FTWE,071,UTCL,Female,1992,34,BUKOMANSIMBI
1,0071607212,NUBUWATI,LWANGA,NATIONAL_ID,CF92098105FTWE,071,UTCL,Female,1992,34,BUKOMANSIMBI
2,0071607213,NUBUWATI,LWANGA,NATIONAL_ID,CF92098105FTWE,071,UTCL,Female,1992,34,BUKOMANSIMBI
3,0071607214,NUBUWATI,LWANGA,NATIONAL_ID,CF92098105FTWE,071,UTCL,Female,1992,34,BUKOMANSIMBI
4,0411671910,DAVID,WASSWA,NATIONAL_ID,CM7305210F16FL,04,UTCL,Male,1973,53,WAKISO


In [8]:
# Keep only local-format MSISDNs (10 digits, leading 0), then convert to 256 format.
# NOTE: this deliberately drops rows already in international 256... format (12 digits);
# widen the filter to .isin([10, 12]) if those should be kept.
df = df[df['msisdn'].astype(str).str.len() == 10].copy()
df_NID = df_NID[df_NID['msisdn'].astype(str).str.len() == 10].copy()

df['msisdn'] = df['msisdn'].astype(str).str.replace(r'^0', '256', regex=True)
df_NID['msisdn'] = df_NID['msisdn'].astype(str).str.replace(r'^0', '256', regex=True)

# Drop unnecessary columns to save memory
df = df.drop(columns=['surname', 'first_name'])
df_NID = df_NID.drop(columns=['surname', 'first_name'])

### Importing MCC Lookup (loaded once, shared by all datasets)

In [9]:
mcc_lu = pd.read_csv(MCC_CSV, dtype=str)

mcc_lu = mcc_lu.rename(columns={"MCC": "mcc", "Country": "country"})
mcc_lu["mcc"] = mcc_lu["mcc"].astype("string").str.strip()
mcc_lu["country"] = mcc_lu["country"].astype("string").str.strip()
mcc_lu = mcc_lu.drop_duplicates(subset=["mcc"])

mcc_lu.head()

,mcc,country
0,289,Abkhazia
1,412,Afghanistan
2,276,Albania
3,603,Algeria
4,544,American Samoa


### Importing GSMA data

In [10]:
gsma_query = """
SELECT
    tac,
    oem,
    brand,
    model,
    marketing_name,
    device_type,
    os_family,
    os_version,
    sim_slots,
    has_2g,
    has_3g,
    has_4g,
    has_5g,
    year_released
FROM ceir.gsma_devices
"""
gsma_df = client.query_df(gsma_query)
len(gsma_df)

290402

In [11]:
gsma_df.head()

,tac,oem,brand,model,marketing_name,device_type,os_family,os_version,sim_slots,has_2g,has_3g,has_4g,has_5g,year_released
0,35697403,Not Known,Not Known,ROWEL K658,,Handheld,,,0,0,0,0,0,0
1,35697404,Not Known,G crown,"G265, G765, G865, G965",,Handheld,,,0,0,0,0,0,0
2,35697405,Not Known,QMobile,Q4,Q4 TV,Handheld,Other,,0,1,0,0,0,2013
3,35697406,Not Known,Apple,iPad mini (A1600),iPad mini 3,Tablet,iOS,8_1,1,1,1,1,0,2014
4,35697407,Not Known,TC,TC F6,,Mobile Phone/Feature phone,,,0,0,0,0,0,0


### Importing Fake IMEIs Table

Fake IMEIs are intentionally **not** merged with GSMA: their TACs are typically
unallocated, so the merge would produce mostly nulls/noise.

In [12]:
fake_query = f"SELECT * FROM ceir_gold.imeis_fake_v {date_filter('imei_first_seen')}"
fake_df = client.query_df(fake_query)
len(fake_df)

253205

In [13]:
fake_df.head()

,imei,last_seen,imei_status,imsi,msisdn,device_type,imei_first_seen,cgi,rat,core_type
0,00000000001900,2026-05-26 19:56:59,W,641010259164637,256708283920,<NA>,2026-05-28 13:05:46,641-01-6012-56112,1,
1,00000000166285,2026-05-22 14:45:45,W,641101970878456,,<NA>,2026-05-25 18:32:33,,,
2,00000000528934,2026-05-23 13:38:49,W,641010409065933,256703588730,<NA>,2026-05-29 05:51:52,641-01-7064-54562,6,
3,00000011111111,2026-05-22 14:15:58,W,641101945475179,256783647367,<NA>,2026-05-22 11:21:03,,,
4,00000996542421,2026-05-22 14:10:43,W,641101971338531,256785993576,<NA>,2026-05-25 06:38:01,,,


In [14]:
# Drop empty device_type column and re-arrange columns
fake_df = fake_df.drop(columns=['device_type'])
fake_df = fake_df[['imei_first_seen', 'last_seen', 'imei', 'imei_status', 'imsi', 'msisdn', 'rat', 'cgi']]

In [15]:
# Enrich: KYC coalesce -> MNO from IMSI where missing -> country from MCC
fake_df = attach_kyc(fake_df, df_NID, df)
fake_df = fill_mno_from_imsi(fake_df)
fake_df = add_country(fake_df, mcc_lu)

In [16]:
fake_df.head()

,imei_first_seen,last_seen,imei,imei_status,imsi,msisdn,rat,cgi,id_type,id_number,prefix,mno,gender,birth_year,age,district,mcc,country
0,2026-05-28 13:05:46,2026-05-26 19:56:59,00000000001900,W,641010259164637,256708283920,1,641-01-6012-56112,NATIONAL_ID,CM93032100587D,070,AIRTEL,Male,1993,33,MUKONO,641,Uganda
1,2026-05-25 18:32:33,2026-05-22 14:45:45,00000000166285,W,641101970878456,,,,NaN,<NA>,NaN,MTN,NaN,<NA>,<NA>,NaN,641,Uganda
2,2026-05-29 05:51:52,2026-05-23 13:38:49,00000000528934,W,641010409065933,256703588730,6,641-01-7064-54562,NATIONAL_ID,CF68049101JA6J,070,AIRTEL,Female,1968,58,MAYUGE,641,Uganda
3,2026-05-22 11:21:03,2026-05-22 14:15:58,00000011111111,W,641101945475179,256783647367,,,NATIONAL_ID,CF6200910496ZC,078,MTN,Female,1962,64,KABALE,641,Uganda
4,2026-05-25 06:38:01,2026-05-22 14:10:43,00000996542421,W,641101971338531,256785993576,,,NATIONAL_ID,CM87004101L5RD,078,MTN,Male,1987,39,BUSHENYI,641,Uganda


### Importing Genuine IMEIs Table

In [17]:
genuine_query = f"SELECT * FROM ceir_gold.imeis_genuine_v {date_filter('imei_first_seen')}"
genuine_df = client.query_df(genuine_query)
len(genuine_df)

3347134

In [18]:
# Drop device_type — regenerated from GSMA TAC data below
genuine_df = genuine_df.drop(columns=['device_type'])

# Derive TAC (first 8 digits of the IMEI) and re-arrange columns
genuine_df['tac'] = genuine_df['imei'].astype(str).str[:8]
genuine_df = genuine_df[['imei_first_seen', 'last_seen', 'tac', 'imei', 'imei_status', 'imsi', 'msisdn', 'rat', 'cgi']]

# Merge with GSMA data to get device details
genuine_df = genuine_df.merge(gsma_df, on='tac', how='left')

In [19]:
# Enrich: KYC coalesce -> MNO from IMSI where missing -> country from MCC
genuine_df = attach_kyc(genuine_df, df_NID, df)
genuine_df = fill_mno_from_imsi(genuine_df)
genuine_df = add_country(genuine_df, mcc_lu)

In [20]:
# Convert GSMA numeric columns to nullable integers (removes decimal points)
int_cols = ['sim_slots', 'has_2g', 'has_3g', 'has_4g', 'has_5g', 'year_released']
genuine_df[int_cols] = genuine_df[int_cols].astype('Int64')

In [21]:
genuine_df.head()

,imei_first_seen,last_seen,tac,imei,imei_status,imsi,msisdn,rat,cgi,oem,...,id_type,id_number,prefix,mno,gender,birth_year,age,district,mcc,country
0,2026-05-23 05:15:02,1970-01-01 00:00:00,00440123,00440123063041,,,,,,Not Known,...,NaN,<NA>,NaN,NaN,NaN,<NA>,<NA>,NaN,,UNKNOWN
1,2026-05-23 20:19:49,2026-05-27 01:36:20,00440245,00440245758171,W,635107016429703,250794370587,6,641-01-1100-29553,Not Known,...,NaN,<NA>,NaN,NaN,NaN,<NA>,<NA>,NaN,635,Rwanda
2,2026-05-29 11:08:18,2026-06-06 18:09:49,00440254,00440254264131,W,630021286018694,,6,641-10-1234-5678,Not Known,...,NaN,<NA>,NaN,NaN,NaN,<NA>,<NA>,NaN,630,Democratic Republic of Congo
3,2026-05-27 17:42:19,2026-05-27 15:46:16,00440254,00440254275167,W,630010637881226,,6,641-10-1234-5678,Not Known,...,NaN,<NA>,NaN,NaN,NaN,<NA>,<NA>,NaN,630,Democratic Republic of Congo
4,2026-05-22 08:03:36,1970-01-01 00:00:00,00440254,00440254275495,,,,,,Not Known,...,NaN,<NA>,NaN,NaN,NaN,<NA>,<NA>,NaN,,UNKNOWN


### Cloned IMEIs

In [22]:
# Filter in SQL so only the analysis month is pulled (the view's timestamp
# column is first_detected_at, renamed to imei_first_seen after loading)
clone_query = f"SELECT * FROM ceir_gold.cloned_imeis_v {date_filter('first_detected_at')}"
clone_df = client.query_df(clone_query)
len(clone_df)

0

In [23]:
# Drop device_type (regenerated from GSMA) and the array columns
clone_df = clone_df.drop(columns=['device_type', 'imsis', 'msisdns'])

# Rename for consistency with the other datasets
clone_df = clone_df.rename(columns={'first_detected_at': 'imei_first_seen', 'last_change_at': 'last_seen'})

# Derive TAC and re-arrange columns
clone_df['tac'] = clone_df['imei'].astype(str).str[:8]
clone_df = clone_df[['imei_first_seen', 'last_seen', 'tac', 'imei', 'imsi_count', 'msisdn_count']]

# Merge with GSMA data to get device details
clone_df = clone_df.merge(gsma_df, on='tac', how='left')

KeyError: "['device_type', 'imsis', 'msisdns'] not found in axis"

In [ ]:
clone_df.head()

### Export — filenames derived from MONTH, nothing to edit

In [ ]:
save_monthly_parquet(fake_df, "Fake")
save_monthly_parquet(genuine_df, "Genuine")
save_monthly_parquet(clone_df, "Cloned")

### Cleanup

In [ ]:
client.close()